# Ethiopia Climate EDA — Task 2

**Objective:** Profile, clean, and perform focused exploratory data analysis on the climate dataset.

> Change `COUNTRY = "Ethiopia"` and `CSV_FILE = "ethiopia.csv"` if your country/file is different.


## 0. Setup


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional but useful for heatmap
try:
    import seaborn as sns
except ImportError:
    sns = None

COUNTRY = "Ethiopia"
CSV_FILE = "ethiopia.csv"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## 1. Data Loading & NASA Header Handling

The assignment says to load using `pd.read_csv("<country>.csv")`.

NASA CSVs may include metadata/header comments before the real column names.  
This cell first tries normal loading. If `YEAR` and `DOY` are not found, it scans for the real header line that starts with `YEAR`.


In [ ]:
def load_climate_csv(path):
    # First: assignment-required simple load
    df_try = pd.read_csv(path)

    if {"YEAR", "DOY"}.issubset(df_try.columns):
        return df_try, 0

    # Fallback: NASA-style header handling
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    header_row = None
    for i, line in enumerate(lines):
        if line.strip().upper().startswith("YEAR"):
            header_row = i
            break

    if header_row is None:
        raise ValueError("Could not find YEAR/DOY columns. Check your CSV header.")

    df = pd.read_csv(path, skiprows=header_row)
    return df, header_row


df_raw, skipped_rows = load_climate_csv(CSV_FILE)

print(f"Loaded shape: {df_raw.shape}")
print(f"NASA/header rows skipped: {skipped_rows}")
df_raw.head()


## 2. Date Parsing

We add:
- `Country`
- `date`
- `Month`
- `T2M_RANGE` for later relationship analysis


In [ ]:
df = df_raw.copy()

df["Country"] = COUNTRY

# Convert YEAR + DOY into proper date
df["date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract month
df["Month"] = df["date"].dt.month

# Useful derived climate variable
if {"T2M_MAX", "T2M_MIN"}.issubset(df.columns):
    df["T2M_RANGE"] = df["T2M_MAX"] - df["T2M_MIN"]

df.head()


## 3. Replace NASA Sentinel Missing Values

NASA uses `-999` as a missing/out-of-range sentinel value.  
We replace all `-999` values with `NaN` before statistics.


In [ ]:
sentinel_count = (df == -999).sum(numeric_only=False).sum()
df = df.replace(-999, np.nan)

print(f"Total -999 sentinel values replaced with NaN: {sentinel_count}")


## 4. Duplicate Rows

Document how many duplicate rows were found and removed.


In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_rows = df[df.duplicated(keep=False)]

print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    display(duplicate_rows.head())

df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after duplicate removal: {df.shape}")


**Duplicate interpretation:**  
Write here after running:

Example:  
`The dataset had X duplicate rows. They were exact duplicate records across all columns, so they were removed to avoid double-counting climate observations.`


## 5. Summary Statistics


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

summary_stats = df[numeric_cols].describe().T
summary_stats


**Summary interpretation:**  
Write 3–5 sentences here after checking the table.

Example points:
- Average temperature (`T2M`) level.
- Range between `T2M_MIN` and `T2M_MAX`.
- Rainfall (`PRECTOTCORR`) variability.
- Wind speed (`WS2M`) normal range.


## 6. Missing Value Report

Show count and percentage of missing values per column.  
List columns with more than 5% missing values.


In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_report


In [ ]:
high_missing = missing_report[missing_report["missing_percent"] > 5]
high_missing


**Missing-value interpretation:**  
Write here after running:

Example:  
`Columns above 5% missing may weaken trend/correlation analysis because a large part of the observations are unavailable. If key variables such as T2M or PRECTOTCORR have high missingness, conclusions about temperature or rainfall should be treated carefully.`


## 7. Outlier Detection Using Z-Scores

Variables required:
- `T2M`
- `T2M_MAX`
- `T2M_MIN`
- `PRECTOTCORR`
- `RH2M`
- `WS2M`
- `WS2M_MAX`

Rows are flagged as outliers where absolute Z-score is greater than 3.


In [ ]:
z_cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]
z_cols = [col for col in z_cols if col in df.columns]

z_scores = pd.DataFrame(index=df.index)

for col in z_cols:
    mean = df[col].mean(skipna=True)
    std = df[col].std(skipna=True)
    if std == 0 or pd.isna(std):
        z_scores[col + "_z"] = np.nan
    else:
        z_scores[col + "_z"] = (df[col] - mean) / std

outlier_mask = (z_scores.abs() > 3).any(axis=1)
outlier_count = outlier_mask.sum()

print(f"Columns checked: {z_cols}")
print(f"Rows flagged as outliers: {outlier_count}")
print(f"Outlier percentage: {outlier_count / len(df) * 100:.2f}%")

df_outliers = df.loc[outlier_mask].copy()
df_outliers.head()


In [ ]:
outlier_counts_by_column = (z_scores.abs() > 3).sum().sort_values(ascending=False)
outlier_counts_by_column


**Outlier decision:**  
Recommended decision: **retain outliers** unless they are clear data errors.

Reasoning you can use:
`Climate data can naturally contain extreme rainfall, heat, cold, humidity, or wind events. Since this assignment studies climate trends, removing real extremes may hide important climate signals. Therefore, I retained outliers after flagging them. Missing values were handled separately.`


## 8. Missing Value Handling / Basic Cleaning

Rule used:
1. Drop rows where more than 30% of values are missing.
2. Forward-fill remaining missing weather values.
3. Back-fill any remaining first-row gaps.


In [ ]:
missing_row_percent = df.isna().mean(axis=1) * 100
rows_over_30_missing = (missing_row_percent > 30).sum()

print(f"Rows with more than 30% missing values: {rows_over_30_missing}")

df_clean = df.loc[missing_row_percent <= 30].copy()

weather_cols = [
    "T2M", "T2M_MAX", "T2M_MIN", "T2M_RANGE",
    "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"
]
weather_cols = [col for col in weather_cols if col in df_clean.columns]

df_clean = df_clean.sort_values("date").reset_index(drop=True)

df_clean[weather_cols] = df_clean[weather_cols].ffill().bfill()

print(f"Cleaned shape: {df_clean.shape}")
df_clean.isna().sum()


**Cleaning decision:**  
Write here:

`Rows with more than 30% missing values were dropped because too much information was unavailable. Remaining weather-variable gaps were forward-filled because daily climate observations are time-series data, so nearby observations are usually more reasonable than replacing with a full-period mean. Back-fill was used only to handle missing values at the beginning of the dataset.`


## 9. Export Cleaned Data

The cleaned CSV is exported to `data/<country>_clean.csv`.

Make sure `data/` is in `.gitignore` so CSV files are not committed to GitHub.


In [ ]:
country_slug = COUNTRY.lower().replace(" ", "_")
clean_path = DATA_DIR / f"{country_slug}_clean.csv"

df_clean.to_csv(clean_path, index=False)
print(f"Cleaned CSV exported to: {clean_path}")


## 10. Time Series Analysis — Monthly Average Temperature


In [ ]:
monthly = (
    df_clean
    .set_index("date")
    .resample("M")
    .agg({
        "T2M": "mean",
        "PRECTOTCORR": "sum"
    })
    .reset_index()
)

monthly["MonthLabel"] = monthly["date"].dt.strftime("%Y-%m")

warmest = monthly.loc[monthly["T2M"].idxmax()]
coolest = monthly.loc[monthly["T2M"].idxmin()]

plt.figure(figsize=(14, 6))
plt.plot(monthly["date"], monthly["T2M"], marker="o", linewidth=1)
plt.title(f"Monthly Average Temperature in {COUNTRY} (2015–2026)")
plt.xlabel("Date")
plt.ylabel("Average T2M Temperature")
plt.grid(True, alpha=0.3)

plt.annotate(
    f"Warmest: {warmest['MonthLabel']}\n{warmest['T2M']:.2f}",
    xy=(warmest["date"], warmest["T2M"]),
    xytext=(warmest["date"], warmest["T2M"] + 1),
    arrowprops=dict(arrowstyle="->")
)

plt.annotate(
    f"Coolest: {coolest['MonthLabel']}\n{coolest['T2M']:.2f}",
    xy=(coolest["date"], coolest["T2M"]),
    xytext=(coolest["date"], coolest["T2M"] - 2),
    arrowprops=dict(arrowstyle="->")
)

plt.tight_layout()
plt.show()


**Temperature trend interpretation:**  
Write here:
- Is temperature increasing, decreasing, or mostly stable?
- Which month/year is warmest?
- Which month/year is coolest?
- Any abnormal spikes or drops?


## 11. Time Series Analysis — Monthly Rainfall


In [ ]:
rainiest = monthly.loc[monthly["PRECTOTCORR"].idxmax()]

plt.figure(figsize=(14, 6))
plt.bar(monthly["date"], monthly["PRECTOTCORR"], width=20)
plt.title(f"Monthly Total Rainfall in {COUNTRY} (2015–2026)")
plt.xlabel("Date")
plt.ylabel("Monthly Total PRECTOTCORR")
plt.grid(axis="y", alpha=0.3)

plt.annotate(
    f"Peak rainfall: {rainiest['MonthLabel']}\n{rainiest['PRECTOTCORR']:.2f}",
    xy=(rainiest["date"], rainiest["PRECTOTCORR"]),
    xytext=(rainiest["date"], rainiest["PRECTOTCORR"] * 1.05),
    arrowprops=dict(arrowstyle="->")
)

plt.tight_layout()
plt.show()


In [ ]:
seasonal_rain = df_clean.groupby("Month")["PRECTOTCORR"].sum().sort_values(ascending=False)
seasonal_rain


**Rainfall interpretation:**  
Write here:
- Which months have the highest rainfall?
- Do they match the expected rainy season?
- Are there drought-like months or abnormal wet months?


## 12. Correlation Heatmap


In [ ]:
corr_cols = df_clean.select_dtypes(include=[np.number]).columns
corr = df_clean[corr_cols].corr()

plt.figure(figsize=(12, 8))

if sns is not None:
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
else:
    plt.imshow(corr)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.colorbar()

plt.title(f"Correlation Heatmap — {COUNTRY}")
plt.tight_layout()
plt.show()


## 13. Scatter Plots


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df_clean["T2M"], df_clean["RH2M"], alpha=0.5)
plt.title(f"T2M vs RH2M — {COUNTRY}")
plt.xlabel("T2M")
plt.ylabel("RH2M")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
if "T2M_RANGE" in df_clean.columns and "WS2M" in df_clean.columns:
    plt.figure(figsize=(8, 6))
    plt.scatter(df_clean["T2M_RANGE"], df_clean["WS2M"], alpha=0.5)
    plt.title(f"T2M_RANGE vs WS2M — {COUNTRY}")
    plt.xlabel("T2M_RANGE")
    plt.ylabel("WS2M")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("T2M_RANGE or WS2M not available.")


In [ ]:
# Find three strongest correlations excluding self-correlation duplicates
corr_pairs = corr.abs().unstack().reset_index()
corr_pairs.columns = ["var1", "var2", "abs_corr"]

corr_pairs = corr_pairs[corr_pairs["var1"] != corr_pairs["var2"]]
corr_pairs["pair"] = corr_pairs.apply(lambda x: tuple(sorted([x["var1"], x["var2"]])), axis=1)
corr_pairs = corr_pairs.drop_duplicates("pair").drop(columns="pair")

strongest_corrs = corr_pairs.sort_values("abs_corr", ascending=False).head(3)
strongest_corrs


**Three strongest correlations interpretation:**  
Write here after running:

Example format:
1. `T2M_MAX` and `T2M` show strong positive correlation because maximum daily temperature rises when average temperature rises.
2. `T2M_MIN` and `T2M` show strong positive correlation because minimum temperature contributes to daily average temperature.
3. `T2M` and `RH2M` may show negative correlation if hotter periods are drier.


## 14. Distribution Analysis — Rainfall Histogram

Rainfall is often heavily skewed because many days have little/no rain and a few days have heavy rain.


In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(df_clean["PRECTOTCORR"].dropna(), bins=40)
plt.title(f"Distribution of Daily Rainfall — {COUNTRY}")
plt.xlabel("PRECTOTCORR")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Log-scale version if rainfall is heavily skewed
rain_log = np.log1p(df_clean["PRECTOTCORR"].dropna())

plt.figure(figsize=(8, 6))
plt.hist(rain_log, bins=40)
plt.title(f"Log-Scaled Distribution of Daily Rainfall — {COUNTRY}")
plt.xlabel("log(1 + PRECTOTCORR)")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Rainfall distribution interpretation:**  
Write here:

`The rainfall distribution is likely right-skewed, meaning most days have low rainfall while a smaller number of days have heavy rainfall. This is common in climate data and explains why a log-scaled histogram may show the pattern more clearly.`


## 15. Bubble Chart — T2M vs RH2M, Bubble Size = PRECTOTCORR


In [ ]:
bubble_size = df_clean["PRECTOTCORR"].fillna(0)

# Scale bubble size so the chart remains readable
bubble_size_scaled = 20 + (bubble_size / bubble_size.max()) * 300 if bubble_size.max() > 0 else 20

plt.figure(figsize=(9, 6))
plt.scatter(
    df_clean["T2M"],
    df_clean["RH2M"],
    s=bubble_size_scaled,
    alpha=0.4
)
plt.title(f"Bubble Chart: Temperature, Humidity, and Rainfall — {COUNTRY}")
plt.xlabel("T2M")
plt.ylabel("RH2M")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Bubble chart interpretation:**  
Write here:
- Do large rainfall bubbles appear mostly during high humidity?
- Are hotter observations linked with lower or higher humidity?
- Are rainy observations clustered or spread out?


## 16. Final Notes / References Used

References:
- pandas `read_csv` documentation for CSV loading and handling commented/header lines.
- pandas `to_datetime` documentation for converting year/day-of-year values into dates.
- pandas missing-value tools: `replace`, `isna`, `ffill`, `bfill`.
- Matplotlib documentation for line, bar, scatter, and histogram plots.
